In [ ]:
from chggen.common.sample_utils import CSP_Generator
from chggen.common.data_utils import mkdir
from types import SimpleNamespace
import numpy as np
from pymatgen.core import Structure, Composition, Element, Lattice


In [ ]:
csp = CSP_Generator(chggen_path = "./files/cut_7_conv_3_epoch=27-val_loss=0.87.ckpt",
                    device='cuda:6')

In [ ]:
ld_kwargs = SimpleNamespace(
        n_step_each = 5,            # Corrector
        min_sigma = 0.01,
        num_noise_level = 200,
        signal_to_noise_ratio = 0.4,
        save_traj = False,
        disable_bar = False,
    )

In [ ]:
gen_kwargs = SimpleNamespace(
        num_gen = 3, # number of structures generated from the cubic lattice
        num_mutation = 2, # number of mutations during the relax-generation iteration
        num_cell = 1, # number of times to the formula
        ehull_cutoff = 0.06,
        )

volume = 24


#  Generate a simple cubic structure via diffusion
s_list_cubic = csp.generate_simple_cubic_structure( comp_str= 'Na2ZrCl6', atom_volume=volume,
                                                   gen_kwargs = gen_kwargs, ld_kwargs = ld_kwargs)
                                                   

In [ ]:
#  Generate seven different bravis lattices via diffusion
s_list_Bravis = csp.generate_structures_from_Bravis(comp_str= 'Na2ZrCl6', atom_volume=volume,
                                                    gen_kwargs=gen_kwargs, ld_kwargs=ld_kwargs)

In [ ]:
mkdir('files/volume_'+ str(volume))

for ii, s in enumerate(s_list_Bravis):
    s.to(filename='files/volume_'+ str(volume)+'/Na2ZrCl4_'+str(ii)+'.cif')

In [ ]:
from chggen.common.sample_utils import get_inpaint_data_fromHost
from chggen.common.sample_utils import get_batch_inpaint_data_fromHost
from chggen.common.sample_utils import get_coarse_grain_framework



In [ ]:
for s in s_list_cubic:
    s.remove_species(['Li'])

In [ ]:
# s_list_cubic

In [ ]:
gen_inputs_batch = get_batch_inpaint_data_fromHost(model=csp.chggen, 
                                       host_structure_list= s_list_cubic,
                                       num_intercalant_list= [1, 1, 1],
                                       species = 'Li',
                                    )

In [ ]:
gen_inputs_batch

In [ ]:
s_inpaint_list= csp.generate_from_host_structure(host_structure_list= s_list_cubic,
                                 num_intercalant_list= [1, 1, 1],
                                 ld_kwargs=ld_kwargs, 
                                 species= 'Li')

In [ ]:
mkdir('files/inpaint_volume_'+ str(volume))

for ii, s in enumerate(s_inpaint_list):
    s.to(filename='files/inpaint_volume_'+ str(volume)+'/LiF_'+str(ii)+'.cif')

In [ ]:
s_CG_frame, symbol_CG, num_species = get_coarse_grain_framework(s_list_Bravis[-1], species_to_remove = 'Li')

In [ ]:
# Generate inpainted structures from the coarse-grained host structures
s_inpaint_list= csp.generate_from_host_structure(host_structure_list= [s_CG_frame]*5,
                                 num_intercalant_list= [1]*5,
                                 ld_kwargs=ld_kwargs, 
                                 species= 'Li')

In [ ]:
ROOT = 'files/volume_'+ str(np.round(s_CG_frame.volume / 2, 2))
mkdir(ROOT)

for ii, s in enumerate(s_inpaint_list):
    s.to(filename=ROOT +'/inpaint_LiF_'+str(ii)+'.cif')